# 예제 02. 계층 쌓기
빅데이터프로그래밍 · 6주차

## 목표
- `nn.Linear` `nn.ReLU` `nn.Sequential` 을 사용한다
- 층을 늘리면 파라미터가 어떻게 늘어나는지 본다
- 활성화 함수가 없으면 층을 쌓아도 소용없음을 확인한다


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)


## 1. Sequential — 순서대로 통과시키기


In [ ]:
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)

print(model)


In [ ]:
x = torch.randn(5, 4)              # 데이터 5개, 특성 4개
print("입력:", x.shape)
print("출력:", model(x).shape)      # (5, 1)


## 2. 층마다 shape이 어떻게 바뀌는가
중간 결과를 하나씩 꺼내 봅니다.


In [ ]:
h = x
for i, layer in enumerate(model):
    h = layer(h)
    print(f"{i} {layer.__class__.__name__:8s} → {tuple(h.shape)}")


## 3. 파라미터 개수 세기
`Linear(입력, 출력)` 의 파라미터는 `입력 × 출력 + 출력` 입니다.


In [ ]:
for name, p in model.named_parameters():
    print(f"{name:12s} {tuple(p.shape)}  {p.numel():4d}개")

total = sum(p.numel() for p in model.parameters())
print("\n전체:", total, "= 4*8+8 + 8*1+1 =", 4*8+8 + 8*1+1)


In [ ]:
# 은닉층 크기를 키우면
for hidden in [8, 32, 128]:
    m = nn.Sequential(nn.Linear(4, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    print(f"hidden {hidden:4d}  파라미터 {sum(p.numel() for p in m.parameters()):6d}개")


## 4. 활성화 함수가 없으면 층을 쌓아도 소용없습니다
선형 변환을 두 번 해도 결국 하나의 선형 변환과 같습니다.


In [ ]:
no_act = nn.Sequential(nn.Linear(2, 4), nn.Linear(4, 1))     # ReLU 없음

# 두 층의 가중치를 곱해 하나로 합칠 수 있습니다
W1, b1 = no_act[0].weight.data, no_act[0].bias.data
W2, b2 = no_act[1].weight.data, no_act[1].bias.data

W_eq = W2 @ W1
b_eq = W2 @ b1 + b2

x = torch.randn(3, 2)
print("2층 모델    :", no_act(x).flatten().data.round(decimals=4))
print("1층으로 합침:", (x @ W_eq.T + b_eq).flatten().round(decimals=4))


→ 값이 같습니다. **활성화 함수가 층을 의미 있게 만듭니다.**


## 5. 층을 늘려 보기


In [ ]:
deep = nn.Sequential(
    nn.Linear(4, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
print(deep)
print("파라미터:", sum(p.numel() for p in deep.parameters()))
print("출력:", deep(torch.randn(5, 4)).shape)


## 6. 자주 만나는 오류 — 층 사이 크기 불일치


In [ ]:
bad = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(16, 1),      # 8이 아니라 16 — 어긋남
)
try:
    bad(torch.randn(5, 4))
except RuntimeError as err:
    print("RuntimeError:", err)


앞 층의 **출력 수**와 뒤 층의 **입력 수**가 같아야 합니다.

## 직접 해보기
1. 입력 10, 은닉 32, 출력 3인 모델을 만들고 파라미터 개수를 세세요.
2. 은닉층을 하나 더 넣으면 파라미터가 몇 개 늘어나나요?


In [ ]:
# 여기에 작성하세요
